MD**Intelligent Vision Systems · Lab 5 · Self-Driving Car Vision: Finding Lane Lines**  
Companion to **Lecture 5 (Part II) — Computer Vision for Self-Driving Cars**  
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_5/05-lane-finding-project.ipynb) &nbsp; [Lecture 5 Part II slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/05_SDC_Lane_Finding.html#/assignment-2-finding-lane-lines) &nbsp; [All Lab 5 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-5-lane-lines)

# Lab 5E · Assignment 2 — Finding Lane Lines on the Road

**This notebook is your Assignment 2 hand-in** (adapted from Project 1 of the Udacity *Self-Driving Car Engineer*
Nanodegree, [CarND-LaneLines-P1](https://github.com/udacity/CarND-LaneLines-P1), MIT licence).

You will write a pipeline that finds lane lines, first on **six test images** and then on **two videos** (a video is just a
series of images). Your output should look like the example in the lecture: one solid line on the left lane and one on the
right, from the bottom of the image towards the horizon.

| Part | What you do | Marks |
|---|---|---|
| 1 | Build `process_image()` from the lecture tools and run it on all test images | 30 |
| 2 | Improve `draw_lines()`: **average** and **extrapolate** to one line per lane | 30 |
| 3 | Run on `solidWhiteRight.mp4` and `solidYellowLeft.mp4`; smooth across frames | 20 |
| 4 | Write-up: pipeline, shortcomings, possible improvements (see the end) | 20 |
| ★ | Optional: make it work on `challenge.mp4` (bonus +10) | — |

**Submit:** this notebook with all outputs + your two output videos (`test_videos_output/`) + the write-up (Markdown cell at the end or `writeup.md`).

In [ ]:
# Setup (only needed on Google Colab or a fresh environment).
# Uncomment the next line to install the packages this notebook uses:
# !pip install -q opencv-python numpy matplotlib imageio imageio-ffmpeg

import os, urllib.request

# Fetch the lab images if they are not next to the notebook.
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_5"
for f in ['exit-ramp.jpg', 'test_images/solidWhiteCurve.jpg', 'test_images/solidWhiteRight.jpg', 'test_images/solidYellowCurve.jpg', 'test_images/solidYellowCurve2.jpg', 'test_images/solidYellowLeft.jpg', 'test_images/whiteCarLaneSwitch.jpg']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{f}", f)
print("data ready")

MD## Helper functions

These are the tools from the lecture, wrapped as functions. Read them — you will use all of them.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2


def grayscale(img):
    """Grayscale transform: returns an image with only one colour channel (input must be RGB)."""
    return cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)


def canny(img, low_threshold, high_threshold):
    """Canny edge detector (note: cv2.Canny does NOT blur internally - blur first)."""
    return cv2.Canny(img, low_threshold, high_threshold)


def gaussian_blur(img, kernel_size):
    """Gaussian smoothing; kernel_size must be odd."""
    return cv2.GaussianBlur(img, (kernel_size, kernel_size), 0)


def region_of_interest(img, vertices):
    """Keep only the region of the image defined by the polygon `vertices` (int32 array); black out the rest."""
    mask = np.zeros_like(img)
    ignore_mask_color = (255,) * img.shape[2] if img.ndim == 3 else 255
    cv2.fillPoly(mask, vertices, ignore_mask_color)
    return cv2.bitwise_and(img, mask)


def draw_lines(img, lines, color=(255, 0, 0), thickness=2):
    """NAIVE version: draws every Hough segment as it is.
    Your job (Part 2) is to replace this with one averaged, extrapolated line per lane."""
    if lines is None:
        return
    for x1, y1, x2, y2 in lines.reshape(-1, 4):
        cv2.line(img, (int(x1), int(y1)), (int(x2), int(y2)), color, thickness)


def hough_lines(img, rho, theta, threshold, min_line_len, max_line_gap):
    """`img` should be the output of a Canny transform. Returns an image with Hough lines drawn."""
    lines = cv2.HoughLinesP(img, rho, theta, threshold, minLineLength=min_line_len, maxLineGap=max_line_gap)
    line_img = np.zeros((img.shape[0], img.shape[1], 3), dtype=np.uint8)
    draw_lines(line_img, lines)
    return line_img


def weighted_img(img, initial_img, alpha=0.8, beta=1.0, gamma=0.0):
    """initial_img * alpha + img * beta + gamma  (both images must be the same shape)."""
    return cv2.addWeighted(initial_img, alpha, img, beta, gamma)

MD## Look at the test images

In [ ]:
import glob
test_files = sorted(glob.glob("test_images/*.jpg"))
fig, ax = plt.subplots(2, 3, figsize=(18, 7))
for a, f in zip(ax.ravel(), test_files):
    a.imshow(mpimg.imread(f)); a.set_title(os.path.basename(f)); a.axis("off")
plt.tight_layout(); plt.show()

MD## Part 1 · Build the lane-finding pipeline

Fill in `process_image()`. Tip: first get it working step by step on **one** image (plot each intermediate result), then wrap it up.

In [ ]:
def process_image(image):
    """Lane-finding pipeline: RGB image in -> RGB image with the lane lines drawn out.

    TODO (Part 1): build the pipeline from the lecture using the helper functions above:
        1. grayscale  (optional extra: colour-select white & yellow in HLS first)
        2. gaussian_blur
        3. canny
        4. region_of_interest  (a trapezoid; express vertices as fractions of width/height)
        5. hough_lines
        6. weighted_img  -> overlay the lines on the original image
    """
    result = np.copy(image)          # <- replace with your pipeline
    return result

In [ ]:
os.makedirs("test_images_output", exist_ok=True)
fig, ax = plt.subplots(2, 3, figsize=(18, 7))
for a, f in zip(ax.ravel(), test_files):
    out = process_image(mpimg.imread(f))
    mpimg.imsave(os.path.join("test_images_output", os.path.basename(f)), out)
    a.imshow(out); a.set_title(os.path.basename(f)); a.axis("off")
plt.tight_layout(); plt.show()

MD## Part 2 · One line per lane: average and extrapolate

The naive `draw_lines()` draws every Hough segment. Redefine it (in the cell below) so it draws **one** solid line per lane:

1. compute each segment's slope `m = (y2 − y1)/(x2 − x1)`; remember **y points down**, so the **left** lane has **m < 0** and the **right** lane **m > 0**;
2. discard near-horizontal segments (e.g. `|m| < 0.4`);
3. average `m` and intercept `b = y − m·x` per side (weighting by segment length helps);
4. extrapolate: for `y_bottom = image height` and `y_top ≈ 0.6·height`, solve `x = (y − b)/m` and draw the line thick.

Because `hough_lines()` calls `draw_lines()`, re-running the Part 1 cell picks up your new version.

In [ ]:
# TODO (Part 2): redefine draw_lines() here so it averages and extrapolates.
# Keep the same signature:  def draw_lines(img, lines, color=(255, 0, 0), thickness=10): ...

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(18, 7))
for a, f in zip(ax.ravel(), test_files):
    a.imshow(process_image(mpimg.imread(f))); a.set_title(os.path.basename(f)); a.axis("off")
plt.tight_layout(); plt.show()

MD## Part 3 · Videos

`process_video()` reads a video frame by frame, applies your `process_image()` and writes `test_videos_output/<name>.mp4`.
The videos are downloaded from the Udacity project repository on first use.

**Smoothing:** lines that jitter from frame to frame look bad (and would make a car's steering twitch). Keep the previous
frame's slope and intercept and blend them, e.g. an exponential moving average `m ← 0.8·m_prev + 0.2·m_new`.

In [ ]:
import imageio.v2 as imageio
from IPython.display import HTML
from base64 import b64encode

VIDEO_URL = "https://raw.githubusercontent.com/udacity/CarND-LaneLines-P1/master/test_videos"
os.makedirs("test_videos", exist_ok=True); os.makedirs("test_videos_output", exist_ok=True)


def process_video(name, frame_fn, max_frames=None):
    """Run `frame_fn` (RGB image -> RGB image) on every frame of test_videos/<name>.mp4."""
    src, dst = f"test_videos/{name}.mp4", f"test_videos_output/{name}.mp4"
    if not os.path.exists(src):
        urllib.request.urlretrieve(f"{VIDEO_URL}/{name}.mp4", src)
    reader = imageio.get_reader(src)
    fps = reader.get_meta_data()["fps"]
    writer = imageio.get_writer(dst, fps=fps, codec="libx264", macro_block_size=1, ffmpeg_log_level="error")
    for i, frame in enumerate(reader):
        if max_frames and i >= max_frames:
            break
        writer.append_data(frame_fn(frame))
    writer.close(); reader.close()
    return dst


def show_video(path, width=640):
    data = b64encode(open(path, "rb").read()).decode()
    return HTML(f'<video width="{width}" controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

In [ ]:
out = process_video("solidWhiteRight", process_image)
show_video(out)

In [ ]:
out = process_video("solidYellowLeft", process_image)
show_video(out)

MD### 🏁 How good is it really?

Factor the line-finding part of your pipeline into `lane_lines(rgb) -> {"left": (x1, y1, x2, y2), "right": ...}` and
score it on 92 labelled frames (clean + hard) with **[07-leaderboard.ipynb](07-leaderboard.ipynb)** — then send your score to the
class leaderboard.

MD### ★ Optional challenge (+10 bonus)

`challenge.mp4` (1280×720) has a **curve**, **shadows**, a **bright concrete patch** and the car's **hood** in view. Make your
pipeline survive it: resolution-independent ROI, HLS colour selection, rejecting outlier slopes, temporal smoothing…

In [ ]:
# out = process_video("challenge", process_image)
# show_video(out)

MD## Part 4 · Write-up (fill in)

**1. Describe your pipeline.** How many steps, which parameters, and how did you modify `draw_lines()`?

*...your answer...*

**2. Identify potential shortcomings** of your current pipeline (e.g. curves, shadows, worn paint, rain, night, lane changes, hills).

*...your answer...*

**3. Suggest possible improvements** (e.g. perspective transform to a bird's-eye view, polynomial fitting, colour spaces, temporal filtering, deep learning).

*...your answer...*